# Missouri Wildfire Susceptibility Modeling, Spatial Validation, and Community Risk Index

This notebook implements the primary statewide wildfire-susceptibility workflow for Missouri. It includes preprocessing of model-ready geodatabase layers, Random Forest classification, 50 km spatial block cross-validation, secondary XGBoost fire-count regression, mapping, and the Community Wildfire Risk Index.

**Data note:** Large source datasets and the project geodatabase are not included in the repository. See `data/README.md` for data sources and preparation details.


In [ ]:
from collections import Counter
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyogrio
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFECV
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GroupKFold, StratifiedKFold, train_test_split
from sklearn.neighbors import NearestNeighbors
from xgboost import XGBRegressor


In [ ]:
# Project paths
# Expected repository layout: notebooks/ and data/ at the repository root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
GDB_PATH = DATA_DIR / "final_tables.gdb"

if not GDB_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {GDB_PATH}. Place the project geodatabase in data/ "
        "or update GDB_PATH to your local data location."
    )


Define gdb path and load the GDB contents, which consist of one geometry feature class and multiple attribute tables, into a dictionary of GeoDataFrames so that we can easily iterate over the dictionary to preprocess and analyze the data later. We also delete GDFs that contain the variables that didn't make it to the final model.

In [ ]:
# List layers in the project geodatabase.
layers = pyogrio.list_layers(GDB_PATH)
for layer in layers:
    print(layer)

# Load each layer into a dictionary of GeoDataFrames/DataFrames.
gdfs = {}
for layer_name, _ in layers:
    gdfs[layer_name] = gpd.read_file(GDB_PATH, layer=layer_name)

print(gdfs.keys())


In [ ]:
print(gdfs["wui_class_areas_SqM_final"].columns)

fishnet = gdfs["missouri_fishnet_featureclass"].copy()

# Compute area from geometry
fishnet["Cell_Area"] = fishnet.geometry.area

wui = gdfs["wui_class_areas_SqM_final"]

# Join cell area to wui table
wui = wui.merge(
    fishnet[["Cell_ID_Stable", "Cell_Area"]],
    on="Cell_ID_Stable",
    how="left"
)

interface_area = (
    wui[
        [
            "High_Dens_Interface",
            "Med_Dens_Interface",
            "Low_Dens_Interface"
        ]
    ]
    .fillna(0)
    .sum(axis=1)
)

intermix_area = (
    wui[
        [
            "High_Dens_Intermix",
            "Med_Dens_Intermix",
            "Low_Dens_Intermix"
        ]
    ]
    .fillna(0)
    .sum(axis=1)
)

wui["Interface_Pct"] = interface_area / wui["Cell_Area"]
wui["Intermix_Pct"] = intermix_area / wui["Cell_Area"]

# Keep only the relevant columns for the WUI table
wui = wui[["Cell_ID_Stable", "Interface_Pct", "Intermix_Pct"]].copy()

gdfs["wui"] = wui

# delete the old WUI df
del gdfs["wui_class_areas_SqM_final"]

# make sure the old wui gdf has been removed
print(gdfs.keys())

calculate the area of each NLCD landcover class per cell, then consolidate the NLCD landcover classes into broader categories for analysis. Then, create a new NLCD dataframe that includes the percentage of area that each broader category occupies within each cell. Then, remove the original NLCD table from the GeoDataFrames dictionary and replace it with the new NLCD dataframe.

In [ ]:
# display the columns of the NLCD land cover areas dataframe
nlcd = gdfs["final_NLCD_landcover_areas_SqM"].copy()
print(nlcd.columns)

# rename the cell_id column to match the standard naming convention for merging
nlcd.rename(columns={"CELL_ID_STABLE": "Cell_ID_Stable"}, inplace=True)

# assign land cover columns to a list for easier manipulation
landcover_cols = [
    "NLCD_water",
    "NLCD_developed_open",
    "NLCD_developed_low",
    "NLCD_developed_medium",
    "NLCD_developed_high",
    "NLCD_barren",
    "NLCD_deciduous_forest",
    "NLCD_evergreen_forest",
    "NLCD_mixed_forest",
    "NLCD_shrub",
    "NLCD_grassland",
    "NLCD_pasture",
    "NLCD_crop",
    "NLCD_woody_wetland",
    "NLCD_herbaceous_wetland"
]

# calculate total area for each row by summing the land cover columns
nlcd["NLCD_total_area"] = nlcd[landcover_cols].sum(axis=1)

# consolidate land cover classes into broader categories for wildfire interaction
nlcd["Water"] = nlcd["NLCD_water"]

nlcd["Developed"] = (
    nlcd["NLCD_developed_open"] +
    nlcd["NLCD_developed_low"] +
    nlcd["NLCD_developed_medium"] +
    nlcd["NLCD_developed_high"]
)

nlcd["Barren"] = nlcd["NLCD_barren"]

nlcd["Forest"] = (
    nlcd["NLCD_deciduous_forest"] +
    nlcd["NLCD_evergreen_forest"] +
    nlcd["NLCD_mixed_forest"]
)

nlcd["Shrub"] = nlcd["NLCD_shrub"]

nlcd["Grassland"] = nlcd["NLCD_grassland"]

nlcd["Agriculture"] = (
    nlcd["NLCD_pasture"] +
    nlcd["NLCD_crop"]
)

nlcd["Wetlands"] = (
    nlcd["NLCD_woody_wetland"] +
    nlcd["NLCD_herbaceous_wetland"]
)


# convert the consolidated land cover areas to percentages of the total area

# create yet another list of the new consolidated land cover classes for easier manipulation
new_classes = [
    "Water",
    "Developed",
    "Barren",
    "Forest",
    "Shrub",
    "Grassland",
    "Agriculture",
    "Wetlands"
]

# calculate the percentage of each consolidated land cover class relative to the total area
for cls in new_classes:
    nlcd[f"{cls}_Pct"] = nlcd[cls] / nlcd["NLCD_total_area"]

# final nlcd dataframe with only the necessary columns for modeling
nlcd = nlcd[
    [
        "Cell_ID_Stable",
        "Water_Pct",
        "Developed_Pct",
        "Barren_Pct",
        "Forest_Pct",
        "Shrub_Pct",
        "Grassland_Pct",
        "Agriculture_Pct",
        "Wetlands_Pct"
    ]
]

# make sure the percentages sum to 1 (or very close to it) for each row
pct_cols = [c for c in nlcd.columns if c.endswith("_Pct")]
nlcd[pct_cols].sum(axis=1).describe()

# round to 4 decimal places
nlcd_pct = nlcd.round(4)


# clean up the gdfs dictionary to remove the original nlcd and wui dataframes

# delete old NLCD df
del gdfs["final_NLCD_landcover_areas_SqM"]

# add the new nlcd_pct df to the gdfs dictionary
gdfs["nlcd_pct"] = nlcd_pct

# preview the keys of the gdfs dictionary to make sure the old df has
print(gdfs.keys())

Some of the GDFs contain columns with identical names, which causes issues when we try to merge them all together. To avoid this, we must locate and rename the columns with identical names so that every column across every GDF has a unique name. The only identical column that should be shared between all GDFs is the "Cell_ID_Stable" column, which is the unique identifier for each cell that we will merge the GDFs on next. We also run a check at the end of the cell to ensure that all GDFs have the standardized "Cell_ID_Stable" column name.

In [ ]:
# empty list to hold all column names from all gdfs
all_columns = []

# add all column names from each gdf to the list
for key, df in gdfs.items():
    all_columns.extend(df.columns.tolist())

# use Counter to find duplicates in the list of all columns
duplicates = [col for col, count in Counter(all_columns).items() if count > 1]

# print the duplicate columns names
print(duplicates)

# print the gdfs that contain the duplicate columns
for col in duplicates:
    print(f"\n{col}:")
    for key, df in gdfs.items():
        if col in df.columns:
            print("  ", key)


# the column "MEAN" is duplicated across multiple gdfs, so we will rename it in each gdf to avoid confusion during the merge
gdfs["elevation_mean_final"] = gdfs["elevation_mean_final"].rename(
    columns={"MEAN": "Elevation_Mean"}
)

gdfs["prism_vpdmax_final"] = gdfs["prism_vpdmax_final"].rename(
    columns={"MEAN": "VPD_Max"}
)

gdfs["prism_ppt_final"] = gdfs["prism_ppt_final"].rename(
    columns={"MEAN": "Precipitation_Mean"}
)

gdfs["prism_tempmax_final"] = gdfs["prism_tempmax_final"].rename(
    columns={"MEAN": "Temp_Max"}
)

gdfs["prism_tempmean_final"] = gdfs["prism_tempmean_final"].rename(
    columns={"MEAN": "Temp_Mean"}
)



# run a duplicate column check to make sure there are no more GDFs with the same column names

all_columns = []

for key, df in gdfs.items():
    all_columns.extend(df.columns.tolist())

duplicates = [col for col, count in Counter(all_columns).items() if count > 1]

print(duplicates)


# only remaining duplicate is cell_id, which we want
# now, make sure every gdf has the same name for the cell_id column, which is "Cell_ID_Stable"
for key, df in gdfs.items():
    if "Cell_ID_Stable" not in df.columns:
        print(f"{key} is missing Cell_ID_Stable")
        print(df.columns.tolist())

Merge all of the GDFs in the GDFs dictionary into a single GDF, which we will use as our main dataframe for analysis. We merge on the "Cell_ID_Stable" column. We also get rid of any columns that will not make it to the final model here.

In [ ]:
# The Great Merge

# define the fishnet df as the master df to merge all other dfs into
master = gdfs["missouri_fishnet_final"].copy()

# use the Cell_ID_Stable column as the index for the master df, using the dictionary
for key, df in gdfs.items():
    if key != "missouri_fishnet_final":
        print(f"Merging {key}...")
        master = master.merge(
            df,
            on="Cell_ID_Stable",
            how="left"
        )

print(master.shape)


# delete the veg2019 column
del master["MEAN_VEG2019PC"]


# preview final dataframe for modeling
master.head()

Now, we will deal with any null values on a case-by-case basis, using whatever method is most appropriate for each column. The weather data nulls are a result of rasterization, so they do not truly represent values of 0 and we instead replace them with their nearest neighbor value. We find nearest neighbors using the geometry of the fishnet feature class. We do the same with the WUI and SVI columns, since missing values in these columns also represented the existing dataset coverage not extending to those cells.

We can simply replace missing values in the road density and population density columns with 0, since a missing value in these columns more often indicates that there are truly no roads or people in that cell. We do the same with the WUI columns, since missing values in these columns also represented the WUI dataset coverage not extending to those cells.

In [ ]:
# check missing values
master.isna().sum().sort_values(ascending=False)


# load fishnet geometry
fishnet_fc = gdfs["missouri_fishnet_featureclass"].copy()

# create centroid coordinates for nearest neighbor imputation
fishnet_centroids = fishnet_fc[["Cell_ID_Stable", "geometry"]].copy()

fishnet_centroids["x"] = fishnet_centroids.geometry.centroid.x
fishnet_centroids["y"] = fishnet_centroids.geometry.centroid.y


# merge centroid coordinates into master dataframe
master = master.merge(
    fishnet_centroids[["Cell_ID_Stable", "x", "y"]],
    on="Cell_ID_Stable",
    how="left"
)


# function for spatial nearest neighbor imputation
def nearest_neighbor_impute(df, columns):

    # separate missing and complete rows
    missing = df[df[columns].isna().any(axis=1)]
    complete = df[df[columns].notna().all(axis=1)]

    print(f"Missing rows for {columns}: {len(missing)}")

    # skip if nothing is missing
    if len(missing) == 0:
        return

    # nearest neighbor model using centroid coordinates
    nn = NearestNeighbors(n_neighbors=1)

    nn.fit(complete[["x", "y"]])

    # find nearest complete cells
    _, nearest_idx = nn.kneighbors(
        missing[["x", "y"]]
    )

    # replace missing values
    for col in columns:
        df.loc[missing.index, col] = (
            complete.iloc[nearest_idx[:, 0]][col].values
        )


# PRISM climate variables
climate_cols = [
    "Temp_Max",
    "Temp_Mean",
    "Precipitation_Mean",
    "VPD_Max"
]
nearest_neighbor_impute(master, climate_cols)

# WUI variables
wui_cols = [
    "Interface_Pct",
    "Intermix_Pct"
]
nearest_neighbor_impute(master, wui_cols)

# SVI variables
svi_cols = [
    "SVI_Per_Cell"
]
nearest_neighbor_impute(master, svi_cols)

# fill true zero values
master["Road_Density"] = master["Road_Density"].fillna(0)
master["Population_Density"] = master["Population_Density"].fillna(0)


# remove helper coordinates
master = master.drop(columns=["x", "y"])


# verify missing values are handled
print(master[climate_cols].isna().sum())
print(master[wui_cols].isna().sum())
print(master[svi_cols].isna().sum())

There is a row in the master dataframe that represents a tiny sliver of a polygon that lies on the border of Missouri and is too small to extract meaningful data from. We remove this row from the dataframe to avoid filling in nonsensical values for the missing data.

In [ ]:
# Inspect the border sliver with missing NLCD coverage.
display(master[master["NLCD_majority"].isna()])

# Remove the known tiny border sliver using its stable cell identifier.
master = master.loc[master["Cell_ID_Stable"] != 3632].copy()

# Check for remaining missing values.
master.isna().sum().sort_values(ascending=False)


Now we create our target variable. First we need to figure out a good cutoff point for low vs elevated wildfire susceptibility using some exploratory data analysis. We choose a cutoff point of 4 historical ignitions based on 4 representing the 75th percentile of wildfire counts in the dataset while also giving us a reasonable number of observations in both the low and elevated wildfire susceptibility classes.

We create a binary target variable for wildfire susceptibility, where 0-3 historical ignitions is considered low wildfire activity and 4 or more historical ignitions is considered elevated wildfire activity. We will use this binary target variable for our modeling.

In [ ]:
# Inspect wildfire-count percentiles used to select the susceptibility threshold.
percentiles = master["wildfire_count"].quantile(
    [0.75, 0.80, 0.85, 0.90, 0.95]
)
print(percentiles)

# Binary wildfire-susceptibility target.
# 0 = 0-3 historical ignitions
# 1 = 4 or more historical ignitions
master["Risk_Binary"] = (master["wildfire_count"] >= 4).astype(int)

print(master["Risk_Binary"].value_counts())
display(master.groupby("Risk_Binary")["wildfire_count"].describe())


Next, we delete any leftover columns from the master dataframe and define a preliminary set of features to perform RFE on. We will use the results of the RFE to determine which features to keep in our final model.

In [ ]:
master.columns.to_list()

del master["NLCD_majority"]
del master["Shape_Length"]
del master["Shape_Area"]
del master["geometry"]


features = [
    "Water_Pct",
    "Developed_Pct",
    "Barren_Pct",
    "Forest_Pct",
    "Shrub_Pct",
    "Grassland_Pct",
    "Agriculture_Pct",
    "Wetlands_Pct",
    "Precipitation_Mean",
    "Temp_Mean",
    "Temp_Max",
    "VPD_Max",
    "Elevation_Mean",
    "MEAN_HUDEN2020",
    "Population_Density",
    "Road_Density",
]


master.head()

Next, we run RFE on a simple Random Forest model to test different combinations of features. Then, we can use those results to modify our feature list. We will exclude highly correlated features from the final feature list.

In [ ]:


# Define X and y
X = master[features]
y = master["Risk_Binary"]

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

selector = RFECV(
    estimator=rf,
    step=1,
    cv=StratifiedKFold(5),
    scoring="f1",
    n_jobs=-1
)

selector.fit(X, y)

print("Optimal number of features:", selector.n_features_)

selected_features = X.columns[selector.support_]
print(selected_features)

While selecting only a subset of features does not improve the model's predictive ability, it is in our best interest to eliminate any redundancy for the sake of computing efficiency and interpretability. We redefine our feature list based on the RFE results, then we visualize the correlation matrix of the features. 

Because Random Forest models are relatively robust to correlated predictors, climate variables were not removed solely based on correlation. A better next step would be to derive new climate variables that are less correlated with each other or simply reduce the number of climate variables used in the model so that decision trees are not being forced to split on the same information multiple times.

In [ ]:
features = [
    "Forest_Pct",
    "Grassland_Pct",
    "Agriculture_Pct",
    "Precipitation_Mean",
    "Temp_Mean",
    "Temp_Max",
    "VPD_Max",
    "Elevation_Mean",
    "Population_Density",
    "Road_Density",
]

plt.figure(figsize=(14, 12))
sns.heatmap(
    master[features].corr(),
    cmap="coolwarm",
    center=0,
    annot=False,
    square=True,
    linewidths=0.5
)

plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

Model 1 - We will use a Random Forest Classifier to model wildfire susceptibility. We will use the features defined above and the binary target variable we created. We split train/test 80/20, define a Random Forest Classifier, and fit the model to the training data. We then evaluate the model on the test data and print out the accuracy score, confusion matrix, and classification report.

In [ ]:
# Run a Random Forest Classifier with the selected features

X = master[features]
y = master["Risk_Binary"]


# train test split with the classic 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


rfb = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    max_features="sqrt",
    random_state=42,
    class_weight="balanced",
    bootstrap=True,
    n_jobs=-1
)

rfb.fit(X_train, y_train)

y_pred = rfb.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))


# ROC AUC score
y_prob = rfb.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, y_prob)
print("\nAUC:", auc)


# precision vs recall balance is tuned to the best of my abilities

### ROC Curve and Random-Split Performance

The ROC curve summarizes the Random Forest classifier's ability to distinguish lower- and elevated-susceptibility cells across classification thresholds. The dashed diagonal represents no-skill classification.


In [ ]:
from sklearn.metrics import roc_curve

# summarize the primary random train/test evaluation
model_performance = pd.DataFrame({
    "Metric": ["Accuracy", "ROC-AUC"],
    "Score": [
        accuracy_score(y_test, y_pred),
        roc_auc_score(y_test, y_prob)
    ]
})

display(model_performance.round(3))

# calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = roc_auc_score(y_test, y_prob)

fig, ax = plt.subplots(figsize=(7, 6))

ax.plot(
    fpr,
    tpr,
    linewidth=2,
    label=f"Random Forest (AUC = {roc_auc:.3f})"
)

# no-skill reference line
ax.plot([0, 1], [0, 1], "--", label="No-skill")

ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve - Random Forest Wildfire Susceptibility")
ax.legend(loc="lower right")
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()


### Spatial Block Cross-Validation

Because a conventional random train-test split can overestimate model performance when nearby observations are spatially dependent, the Random Forest classifier is also evaluated using five-fold spatial block cross-validation. Grid cells are assigned to 50 km geographic blocks, and entire blocks are held out together during validation.

In [ ]:
# Merge feature table with geometry
split_df = gdfs["missouri_fishnet_featureclass"][
    ["Cell_ID_Stable", "geometry"]
].merge(
    master,
    on="Cell_ID_Stable",
    how="inner"
)

split_df = split_df.set_geometry("geometry")

# Centroids
split_df["X"] = split_df.geometry.centroid.x
split_df["Y"] = split_df.geometry.centroid.y

block_size = 50000  # 50 km

xmin = split_df["X"].min()
ymin = split_df["Y"].min()

split_df["block_x"] = ((split_df["X"] - xmin) // block_size).astype(int)
split_df["block_y"] = ((split_df["Y"] - ymin) // block_size).astype(int)

split_df["block"] = (
    split_df["block_x"].astype(str)
    + "_"
    + split_df["block_y"].astype(str)
)

# assign groups by block
groups = split_df["block"]

gkf = GroupKFold(n_splits=5)



In [ ]:
# Run a Random Forest Classifier with the selected features

accs = []
aucs = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(split_df, groups=groups), 1):

    train = split_df.iloc[train_idx]
    test = split_df.iloc[test_idx]

    X_train = train[features]
    y_train = train["Risk_Binary"]

    X_test = test[features]
    y_test = test["Risk_Binary"]


    model = RandomForestClassifier(
        n_estimators=300,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:,1]

    acc = accuracy_score(y_test, pred)
    auc = roc_auc_score(y_test, prob)

    accs.append(acc)
    aucs.append(auc)

    print(f"Fold {fold}")
    print(f"Accuracy = {acc:.3f}")
    print(f"AUC = {auc:.3f}")
    print()


# precision vs recall balance is tuned to the best of my abilities

### Spatial Cross-Validation Results

The figure below compares accuracy and ROC-AUC across the five 50 km spatial-block folds. Similar performance across geographically separated folds provides a more conservative check of model generalizability than the random train/test split alone.


In [ ]:
# summarize spatial cross-validation results
cv_results = pd.DataFrame({
    "Fold": [f"Fold {i}" for i in range(1, len(accs) + 1)],
    "Accuracy": accs,
    "ROC-AUC": aucs
})

display(cv_results.round(3))

print(f"Mean spatial CV accuracy: {np.mean(accs):.3f}")
print(f"Mean spatial CV ROC-AUC: {np.mean(aucs):.3f}")
print(f"ROC-AUC range: {np.min(aucs):.3f} - {np.max(aucs):.3f}")

# grouped bar chart for fold-level performance
ax = cv_results.set_index("Fold")[["Accuracy", "ROC-AUC"]].plot(
    kind="bar",
    figsize=(8, 6)
)

ax.set_title("50 km Spatial Block Cross-Validation")
ax.set_xlabel("")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Metric")
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()


The classifier is designed to distinguish lower- from elevated-susceptibility cells rather than predict the exact number of fires. Its performance therefore supports susceptibility mapping, while exact wildfire-frequency prediction is evaluated separately with XGBoost regression. The next section examines Random Forest feature importance to identify the strongest predictors of susceptibility.

In [ ]:
# RF feature importance binary

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rfb.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance.head(20)

Next, we will create an XGBoost regressor to test a continuous prediction of wildfire counts. We will use the same features as the binary classifier, but we will not use the binary target variable. Instead, we will use the original wildfire counts as our target variable.

In [ ]:
# X and continuous target
X = master[features]
y = master["wildfire_count"]


# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# XGBoost regression model
xgb_reg = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)


# Train
xgb_reg.fit(X_train, y_train)


# Predict
y_pred = xgb_reg.predict(X_test)


# Evaluate
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R²:", r2_score(y_test, y_pred))


# Compare highest-fire cells
comparison = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})

print("\nHighest actual wildfire counts:")
print(
    comparison.sort_values(
        "Actual",
        ascending=False
    ).head(15)
)


# Correlation between actual and predicted
print("\nCorrelation:")
print(comparison.corr())

The regressor struggles to truly predict the counts of wildfires in each cell, but it does accurately capture the pattern of elevated wildfire susceptibility, which we will map to visualize further. Next, we will analyze feature importance to see which variables are most important in predicting wildfire susceptibility using the XGBoost regressor.

In [ ]:
# extract feature importance
importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": xgb_reg.feature_importances_
})

# sort highest to lowest
importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance)

Next, we will map the original wildfire counts, binary random forest classifier predictions, and XGBoost regressor predictions to visualize the spatial patterns of wildfire susceptibility across Missouri. We will use the same color scheme for all three maps to make it easier to compare the results.

In [ ]:
# Map the 3 models against the actual wildfire counts in each grid cell

# Load fishnet geometry
fishnet = gpd.read_file(
    GDB_PATH,
    layer="missouri_fishnet_featureclass"
)

# Check IDs match
print(fishnet.columns)
print(master.columns)


# Binary classifier predicted risk class
# Uses final selected feature set
master["Predicted_Risk_Class"] = rfb.predict(
    master[features]
)


# Predicted wildfire counts from XGBoost regressor
master["Predicted_Wildfire_Count"] = xgb_reg.predict(
    master[features]
)


# Merge model outputs with fishnet geometry
map_df = fishnet.merge(
    master[
        [
            "Cell_ID_Stable",
            "wildfire_count",
            "Predicted_Risk_Class",
            "Predicted_Wildfire_Count"
        ]
    ],
    on="Cell_ID_Stable",
    how="left"
)

In [ ]:
# map 1 - observed wildfire counts

fig, ax = plt.subplots(figsize=(10, 10))

map_df.plot(
    column="wildfire_count",
    cmap="OrRd",
    legend=True,
    scheme="naturalbreaks",
    k=5,
    ax=ax
)

# Draw outer study-area boundary on top
boundary = gdfs["missouri_fishnet_featureclass"].dissolve()
boundary.boundary.plot(
    ax=ax,
    color="black",
    linewidth=1.2
)

ax.set_title(
    "Historical Wildfire Ignition Counts (30 Years)",
    fontsize=14,
    pad=12
)

ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Map 2 - binary classifier

fig, ax = plt.subplots(figsize=(10,10))

map_df.plot(
    column="Predicted_Risk_Class",
    cmap="OrRd",
    legend=True,
    categorical=True,
    ax=ax
)

# Draw outer study-area boundary on top
boundary = gdfs["missouri_fishnet_featureclass"].dissolve()
boundary.boundary.plot(
    ax=ax,
    color="black",
    linewidth=1.2
)

ax.set_title("Predicted Elevated Wildfire Ignition Susceptibility")
ax.axis("off")

plt.show()

In [ ]:
# Map 3 - xgboost regression wildfire counts predictions

fig, ax = plt.subplots(figsize=(10,10))

map_df.plot(
    column="Predicted_Wildfire_Count",
    cmap="OrRd",
    legend=True,
    scheme="naturalbreaks",
    k=5,
    ax=ax
)

# Draw outer study-area boundary on top
boundary = gdfs["missouri_fishnet_featureclass"].dissolve()
boundary.boundary.plot(
    ax=ax,
    color="black",
    linewidth=1.2
)

ax.set_title("Predicted Wildfire Ignition Frequency")
ax.axis("off")

plt.show()

Next, we can map the residual values of the XGBoost regressor to visualize where the model is overpredicting and underpredicting wildfire counts. This will help us identify areas where the model may be struggling to accurately predict wildfire susceptibility and may require further investigation or additional features to improve performance.

In [ ]:
# map where the regressor is underpredicting fires the most (highest residual cells)


master["Predicted_Count"] = xgb_reg.predict(
    master[xgb_reg.get_booster().feature_names]
)

master["Residual"] = (
    master["wildfire_count"]
    -
    master["Predicted_Count"]
)


master.sort_values(
    "Residual",
    ascending=False
)[
    [
        "Cell_ID_Stable",
        "wildfire_count",
        "Predicted_Count",
        "Residual"
    ]
].head(20)


# add the residuals of each cell back to the fishnet
residual_map = fishnet.merge(
    master[
        [
            "Cell_ID_Stable",
            "wildfire_count",
            "Predicted_Count",
            "Residual"
        ]
    ],
    on="Cell_ID_Stable",
    how="left"
)



# map the residuals
fig, ax = plt.subplots(figsize=(10,10))

residual_map.plot(
    column="Residual",
    cmap="RdBu_r",
    legend=True,
    scheme="naturalbreaks",
    k=5,
    ax=ax
)

ax.set_title("Wildfire Ignition Model Residuals\n(Positive = More Fires Than Predicted)")
ax.axis("off")

plt.show()

Additional assessment: Community Risk Index

The CRI will be calculated using the following features: proportion of WUI per cell area,  SVI calculated by weighting overlapping census polygon SVI scores by their estimated population contribution within each fishnet cell, and predicted elevated wildfire susceptibility from the Random Forest Classifier.

In [ ]:
# Store the probabilities of the elevated wildfire risk class in a new hazard column
X = master[features]
master["Susceptibility_Probability"] = rfb.predict_proba(X)[:, 1]

# average the wui interface and intermix proportions to create a single WUI exposure variable
master["WUI_Exposure"] = (
    master["Interface_Pct"] +
    master["Intermix_Pct"]
) / 2

# SVI per cell is already processed and ready to use, calculated as follows:
# SVI_Per_Cell = sum(SVI_score * estimated population overlap) / sum(estimated population overlap)

# add the CRI features to a new dataframe for use in the Community Risk Index
community_risk_df = master[
    [
        "Cell_ID_Stable",
        "Susceptibility_Probability",
        "WUI_Exposure",
        "SVI_Per_Cell"
    ]
].copy()

community_risk_df.describe()

The three Community Wildfire Risk Index components are represented on comparable 0–1 scales. The index is calculated multiplicatively so that high community risk occurs where wildfire susceptibility, WUI exposure, and social vulnerability overlap.

In [ ]:
community_risk_df["Community_Risk_Index"] = (
    community_risk_df["Susceptibility_Probability"] *
    community_risk_df["WUI_Exposure"] *
    community_risk_df["SVI_Per_Cell"]
)

community_risk_df.sort_values(
    "Community_Risk_Index",
    ascending=False
).head(10)

The CRI indicates that each of its contributing factors (wildfire susceptibility, WUI exposure, and social vulnerability) are important in determining overall community risk. The CDI can be used to identify areas where communities may be at higher risk of wildfire impacts and can inform decision-making for wildfire management and mitigation efforts, not replacing but instead complementing raw wildfire counts and predictions.

Next, we map this relationship to visualize the spatial patterns of community risk across Missouri. We will use the same color scheme for all three maps to make it easier to compare the results. The Community Wildfire Risk Index was log-transformed for visualization due to strong right-skew in the index distribution, only for visualization purposes.

In [ ]:
# log transform for visualization only
community_risk_df["Risk_Index_Log"] = np.log1p(
    community_risk_df["Community_Risk_Index"]
)

# map the Community Risk Index to the fishnet geometry for visualization
risk_map = fishnet_fc.merge(
    community_risk_df[
        [
            "Cell_ID_Stable",
            "Community_Risk_Index",
            "Risk_Index_Log"
        ]
    ],
    on="Cell_ID_Stable",
    how="left"
)

# map the Community Risk Index to visualize areas of elevated community wildfire risk
fig, ax = plt.subplots(figsize=(10,10))

risk_map.plot(
    column="Risk_Index_Log",
    cmap="OrRd",
    legend=True,
    scheme="naturalbreaks",
    k=5,
    ax=ax
)

# Draw outer study-area boundary on top
boundary = gdfs["missouri_fishnet_featureclass"].dissolve()
boundary.boundary.plot(
    ax=ax,
    color="black",
    linewidth=1.2
)

ax.set_title("Community Wildfire Risk Index")
ax.axis("off")

plt.show()

We can see, through raw numbers and visualization, that the CRI is learning from its inputs, not simply one powerful input variable. If a policy maker were to analyze the CRI map to determine where to allocate resources for wildfire mitigation, they might see that the areas of highest wildfire susceptibility are not necessarily the same as the areas of highest WUI exposure or social vulnerability. The CRI can help identify areas where hazard, exposure, and vulnerability intersect, which may be the most critical areas to focus on for wildfire mitigation efforts.

Geographically, the CRI is highest in southern and central Ozark regions of Missouri. These areas are indicated to be at the highest risk to residents and communities from wildfire impacts.